# 01 — Extract: Supabase → Landing

Cria os schemas e o Volume, extrai as tabelas do **Supabase** (via JDBC) e grava na camada **Landing** (CSV no Volume + tabelas).

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS landing;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

CREATE VOLUME IF NOT EXISTS workspace.landing.dados;

In [ ]:
SUPABASE_HOST = "aws-1-us-west-2.pooler.supabase.com"
SUPABASE_PORT = "5432"
SUPABASE_DATABASE = "postgres"
SUPABASE_USER = "postgres.cffpopikxclgtbuyyzgr"
SUPABASE_PASSWORD = "I8ntHq1cKA1mCJsF"

jdbc_url = (
    f"jdbc:postgresql://{SUPABASE_HOST}:"
    f"{SUPABASE_PORT}/{SUPABASE_DATABASE}"
    "?sslmode=require"
)

def read_supabase_table(table_name):
    return (
        spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", f"public.{table_name}")
            .option("user", SUPABASE_USER)
            .option("password", SUPABASE_PASSWORD)
            .option("driver", "org.postgresql.Driver")
            .load()
    )

In [ ]:
df_test = read_supabase_table("clientes")
df_test.show()

In [ ]:
from datetime import datetime

TABLES = [
    "clientes",
    "produtos",
    "pedidos"
]

LANDING_SCHEMA = "landing"
LANDING_BASE_PATH = "/Volumes/workspace/landing/dados"
EXTRACTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("Tabelas para extrair:")
for table in TABLES:
    print(f"- {table}")

print(f"Landing path: {LANDING_BASE_PATH}")
print(f"Data da extracao: {EXTRACTION_DATE}")

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

results = []

for table in TABLES:
    print(f"Extraindo tabela: {table}")

    df = read_supabase_table(table)

    df_landing = (
        df
        .withColumn("_source_table", lit(table))
        .withColumn("_extracted_at", current_timestamp())
    )

    output_path = f"{LANDING_BASE_PATH}/{table}"

    (
        df_landing.write
            .mode("overwrite")
            .option("header", "true")
            .csv(output_path)
    )

    count = df_landing.count()
    columns_count = len(df_landing.columns)

    results.append({
        "table": table,
        "rows": count,
        "columns": columns_count,
        "path": output_path
    })

    print(f"  OK: {count} registros | {columns_count} colunas | {output_path}")

print("Extracao finalizada.")

In [ ]:
for item in results:
    table = item["table"]
    path = item["path"]

    df = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(path)
    )

    (
        df.write
            .mode("overwrite")
            .saveAsTable(f"{LANDING_SCHEMA}.{table}")
    )

    print(f"Tabela criada: {LANDING_SCHEMA}.{table}")

In [ ]:
%sql
SHOW SCHEMAS IN workspace;